Copyright (c) MONAI Consortium  
Licensed under the Apache License, Version 2.0 (the "License");  
you may not use this file except in compliance with the License.  
You may obtain a copy of the License at  
    http://www.apache.org/licenses/LICENSE-2.0  
Unless required by applicable law or agreed to in writing, software  
distributed under the License is distributed on an "AS IS" BASIS,  
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.  
See the License for the specific language governing permissions and  
limitations under the License.  

# MONAI 3D Segmentation with PyTorch 2.0 Features

This tutorial shows how to use `torch.compile` with a segmentation model written in MONAI

NOTE:
`torch.compile` in PyTorch 2.0 requires a full CUDA installation. For CUDA 11.7, please refer to [the NVIDIA website](https://developer.nvidia.com/cuda-11-7-0-download-archive) for the installation guide.
    


## Set up environment

Below we install the latest nightly build of PyTorch and MONAI 0.9.0. For later MONAI versions, the new graph compiler in PyTorch is unable to recognize the `MetaTensor` type, which is a superset of the `torch.Tensor`, in the compilation pipeline.

In [1]:
!python -c "import pkg_resources; pkg_resources.require('torch>1.13'); import torch" || pip3 install numpy --pre torch[dynamo] --force-reinstall --extra-index-url https://download.pytorch.org/whl/nightly/cu117
!python -c "import pkg_resources; pkg_resources.require('monai==0.9.0'); import monai" || (pip3 uninstall -y monai && pip3 install monai[nibabel]==0.9.0)

## Set up imports

In [2]:
import os
import tempfile
from glob import glob

import nibabel as nib
import numpy as np
import torch

import monai
from monai.data import ImageDataset, DataLoader, create_test_image_3d
from monai.transforms import (
    AddChannel,
    Compose,
    RandSpatialCrop,
    ScaleIntensity,
    EnsureType,
)

import warnings
warnings.filterwarnings("ignore", category=UserWarning) 

## Set up timing and training functions

For best accuracies, we use CUDA events and synchronization to measure the forward and backward propagations in training

In [3]:
def timed(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    torch.cuda.synchronize()
    return result, start.elapsed_time(end) / 1000

def train(model, inputs, labels):
    outputs = model(inputs)
    loss_function = monai.losses.DiceLoss(sigmoid=True)
    loss = loss_function(outputs, labels)
    loss.backward()
    return loss

    


## Set up data directory and generate simulated datasets

In [5]:
tempdir = tempfile.mkdtemp()
for i in range(20):
    im, seg = create_test_image_3d(128, 128, 128, num_seg_classes=1)

    n = nib.Nifti1Image(im, np.eye(4))
    nib.save(n, os.path.join(tempdir, f"im{i:d}.nii.gz"))

    n = nib.Nifti1Image(seg, np.eye(4))
    nib.save(n, os.path.join(tempdir, f"seg{i:d}.nii.gz"))

images = sorted(glob(os.path.join(tempdir, "im*.nii.gz")))
segs = sorted(glob(os.path.join(tempdir, "seg*.nii.gz")))

## Set up DataLoader and Train Transforms

In [6]:
# define transforms for image and segmentation
train_imtrans = Compose(
    [ScaleIntensity(), AddChannel(), RandSpatialCrop((96, 96, 96), random_size=False), EnsureType()]
)
train_segtrans = Compose(
    [AddChannel(), RandSpatialCrop((96, 96, 96), random_size=False), EnsureType()]
)
train_segtrans = Compose(
    [AddChannel(), RandSpatialCrop((96, 96, 96), random_size=False), EnsureType()]
)

# define image dataset, data loader
check_ds = ImageDataset(images, segs, transform=train_imtrans, seg_transform=train_segtrans)
check_loader = DataLoader(check_ds, batch_size=10, num_workers=2, pin_memory=torch.cuda.is_available())
im, seg = monai.utils.misc.first(check_loader)
print("Image and Label Shapes: ", im.shape, seg.shape)

# create a training data loader
train_ds = ImageDataset(images, segs, transform=train_imtrans, seg_transform=train_segtrans)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=8, pin_memory=torch.cuda.is_available())

# create UNet, DiceLoss and Adam optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Image and Label Shapes:  torch.Size([10, 1, 96, 96, 96]) torch.Size([10, 1, 96, 96, 96])


## Create network and optimizer

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = monai.networks.nets.BasicUNet(features=(4, 8, 8, 16, 16, 32),  # works
# model = monai.networks.nets.SegResNet(  # works
# model = monai.networks.nets.UNet(channels=(4, 8, 16, 32, 64), strides = (2, 2, 2, 2),  # not working
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), 1e-3)



BasicUNet features: (4, 8, 8, 16, 16, 32).


## A eager mode training

In [ ]:
model.train()
num_epochs = 10

In [14]:
# start a typical PyTorch training
epoch_loss_values = []
eager_time = []
for epoch in range(num_epochs):
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        loss, train_time = timed(lambda: train(model, inputs, labels))
        # loss, train_time = timed(lambda: train(model, inputs, labels))
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        eager_time.append(train_time)
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1}/{num_epochs} average loss: {epoch_loss:.4f}")
print(f"median training time per iteration: {np.median(eager_time):.4f} seconds")


epoch 1/10 average loss: 0.0581
epoch 2/10 average loss: 0.0540
epoch 3/10 average loss: 0.0500
epoch 4/10 average loss: 0.0512
epoch 5/10 average loss: 0.0432
epoch 6/10 average loss: 0.0460
epoch 7/10 average loss: 0.0394
epoch 8/10 average loss: 0.0386
epoch 9/10 average loss: 0.0392
epoch 10/10 average loss: 0.0364
median training time per iteration: 0.1612 seconds


In [15]:
train_opt = torch.compile(train, mode="reduce-overhead")

In [16]:
# start a typical PyTorch training
epoch_loss_values = []
optimize_time = []
for epoch in range(num_epochs):
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        loss, train_time = timed(lambda: train_opt(model, inputs, labels))
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        optimize_time.append(train_time)
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1}/{num_epochs} average loss: {epoch_loss:.4f}")
print(f"median training time per iteration: {np.median(optimize_time):.4f} seconds")


[2022-12-12 14:20:13,312] torch._inductor.graph: [WARNING] Creating implicit fallback for:
  target: aten.max_pool3d_with_indices.default
  args[0]: TensorBox(StorageBox(
    ComputedBuffer(name='buf21', layout=FlexibleLayout('cuda', torch.float32, size=(4, 4, 96, 96, 96), stride=[3538944, 884736, 9216, 96, 1]), data=Pointwise(
      'cuda',
      torch.float32,
      where(load(buf20, i4 + 96 * i3 + 9216 * i2 + 884736 * i1 + 3538944 * i0) > constant(0, torch.float32), load(buf20, i4 + 96 * i3 + 9216 * i2 + 884736 * i1 + 3538944 * i0), load(buf20, i4 + 96 * i3 + 9216 * i2 + 884736 * i1 + 3538944 * i0) * constant(0.1, torch.float32)),
      ranges=(4, 4, 96, 96, 96),
      origins={add_2, repeat_2, unsqueeze_6, primals_5, unsqueeze_9, repeat_3, var_1, primals_6, mul_3, unsqueeze_8, where, add_3, gt_1, rsqrt_1, primals_8, sub_1, where_1, unsqueeze_11, convolution_1, unsqueeze_10, mean_1, view_4, mul_5, mul_4, primals_7, view_3, unsqueeze_7}
    ))
  ))
  args[1]: [2, 2, 2]
  args[2]: [2,

epoch 1/10 average loss: 0.0364
epoch 2/10 average loss: 0.0301
epoch 3/10 average loss: 0.0309
epoch 4/10 average loss: 0.0289
epoch 5/10 average loss: 0.0289
epoch 6/10 average loss: 0.0302
epoch 7/10 average loss: 0.0254
epoch 8/10 average loss: 0.0234
epoch 9/10 average loss: 0.0235
epoch 10/10 average loss: 0.0216
median training time per iteration: 0.1443 seconds
